[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol1/blob/main/notebooks/ch20_code.ipynb)

# 第20章　医療画像の座標系と位置合わせ：経過比較とマルチモダリティ

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol1

## 20.3　ボクセル間隔をそろえる：リサンプリング

In [ ]:
import numpy as np

spacing = img.header.get_zooms()[:3]        # 各軸のボクセル間隔（mm）
if not all(np.isfinite(s) and s > 0 for s in spacing):   # assert は python -O で消えるので、例外で止める
    raise ValueError(f"不正なspacing: {spacing}")

## 数字で追う：リサンプリングに用いる線形補間の重み

In [ ]:
import numpy as np
Q = np.array([[10., 20.], [30., 40.]])

def bilinear(Q, r, c):        # (r, c) は元の格子上の実数座標
    r0, c0 = int(np.floor(r)), int(np.floor(c))
    b, a = r - r0, c - c0     # 基準格子点からの行・列の相対位置(0〜1)
    return (Q[r0, c0]*(1-a)*(1-b) + Q[r0, c0+1]*a*(1-b)
          + Q[r0+1, c0]*(1-a)*b   + Q[r0+1, c0+1]*a*b)

print(bilinear(Q, 0.3, 0.7))   # 23.0

In [ ]:
from scipy.ndimage import zoom
# 面内 0.7mm → 1.0mm へ。倍率 = 0.7/1.0 = 0.7（画素数は減る）。軸の順は (x, y, z)、
# スライス方向（z）の間隔は変えないので倍率 1.0。軸ごとに「元の間隔 / 目標の間隔」を渡す
img_rs   = zoom(image, zoom=(0.7, 0.7, 1.0), order=1)   # 画像は線形補間
label_rs = zoom(label, zoom=(0.7, 0.7, 1.0), order=0)   # ラベルは最近傍(整数IDを保持する)

In [ ]:
from monai.transforms import Spacingd

Spacingd(keys=["image", "label"], pixdim=(1, 1, 1),
         mode=("bilinear", "nearest"))     # 画像は滑らかに、ラベルは最近傍で

## 具体例で見る：ボクセル間隔を無視すると、何が起きるか

In [ ]:
# 1ボクセルの体積を1mm³にそろえる。以後はボクセル数の値が体積(mm³)の値と一致する。
from monai.transforms import Spacingd

tf = Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0),
              mode=("bilinear", "nearest"))   # 画像は線形、整数IDのラベルは最近傍（直接補間する例）

## 場面によって、必要な処理が大きく異なる

In [ ]:
import SimpleITK as sitk

def phase_shift_mm(fixed_path, moving_path):
    """剛体位置合わせで推定した平行移動成分の大きさ(mm)。局所のずれ全体ではない。"""
    fixed  = sitk.ReadImage(fixed_path,  sitk.sitkFloat32)   # 例：門脈相
    moving = sitk.ReadImage(moving_path, sitk.sitkFloat32)   # 例：動脈相
    reg = sitk.ImageRegistrationMethod()
    reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    reg.SetOptimizerAsRegularStepGradientDescent(1.0, 1e-4, 200)
    reg.SetOptimizerScalesFromPhysicalShift()     # 回転と平行移動のスケールを調整する（後述）
    reg.SetInitialTransform(sitk.CenteredTransformInitializer(
        fixed, moving, sitk.Euler3DTransform(),
        sitk.CenteredTransformInitializerFilter.GEOMETRY))
    reg.SetInterpolator(sitk.sitkLinear)
    tf = reg.Execute(fixed, moving)
    tx, ty, tz = tf.GetParameters()[3:6]          # 平行移動の3成分[mm]
    return (tx**2 + ty**2 + tz**2) ** 0.5         # 平行移動成分の大きさ[mm]

```text
患者A ─┐
患者B ─┼→ 同じアトラス空間へ → 各患者の画像が共通座標系で対応づけられる
患者C ─┘
```

## 位置合わせを必要としない比較もある

```text
前回のCT  →  セグメンテーション  →  前回のマスク  →  体積 12 mL
今回のCT  →  セグメンテーション  →  今回のマスク  →  体積 18 mL
                                              → 6 mL 増えた（位置合わせ不要）
```

## 類似度を用いた変換パラメータの最適化

In [ ]:
import SimpleITK as sitk

fixed  = sitk.ReadImage("ct_今回.nii.gz", sitk.sitkFloat32)   # 基準にする画像（動かさない）
moving = sitk.ReadImage("ct_前回.nii.gz", sitk.sitkFloat32)   # こちらを動かして合わせる

reg = sitk.ImageRegistrationMethod()
# CT同士だが、造影相による画素値の違いを考慮して相互情報量を使う
reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)  # 似ている度合いの指標
reg.SetOptimizerAsRegularStepGradientDescent(learningRate=1.0, minStep=1e-4,
                                             numberOfIterations=200)
reg.SetOptimizerScalesFromPhysicalShift()       # ★回転と平行移動のスケールを調整する（下記）
reg.SetShrinkFactorsPerLevel([4, 2, 1])         # 粗い解像度から順に合わせる（多重解像度）
reg.SetSmoothingSigmasPerLevel([2, 1, 0])
reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()
reg.SetInitialTransform(sitk.CenteredTransformInitializer(
    fixed, moving, sitk.Euler3DTransform(),                       # 剛体（移動＋回転の6自由度）
    sitk.CenteredTransformInitializerFilter.GEOMETRY))
reg.SetInterpolator(sitk.sitkLinear)

transform = reg.Execute(fixed, moving)          # 変換パラメータを推定する
print(transform.GetParameters())                 # 例: 回転3成分 + 平行移動3成分

# 求めた変換で、前回の画像を今回の空間へ移す
moved = sitk.Resample(moving, fixed, transform, sitk.sitkLinear, -1024.0)   # 視野外はCTの空気相当で埋める（0 は水で、偽の構造が出る）
diff  = sitk.GetArrayFromImage(fixed) - sitk.GetArrayFromImage(moved)   # ← 同じグリッド上の差分。造影差・ノイズ・残存する位置ずれにも注意